# Protein Interaction V1 Sample Test Notebook

This notebook runs a tiny end-to-end smoke test for the new protein-level interaction pipeline:
1. Build tiny pair index splits
2. Sanity-check one batched sample
3. Run quick training (few epochs)
4. Run quick evaluation and inspect outputs

All compute stages are submitted through Slurm wrappers (same entry points for small and large runs):
- `sbatch run_build_pairs.slurm --config <yaml> --tiny`
- `sbatch run_train_v1.slurm --config <yaml>`
- `sbatch run_eval_v1.slurm --config <yaml> --split test`

Expected outputs are written under `masif_seed_search/data/protein_interaction_nn/`.

In [1]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys
import time


repo_root = !git rev-parse --show-toplevel
repo_root = Path(repo_root[0])
seed_source = repo_root / "masif_seed_search" / "source"
masif_source = repo_root / "masif" / "source"
protein_interaction_nn_dir = repo_root / "masif_seed_search" / "data" / "protein_interaction_nn"
os.environ["PYTHONPATH"] = f"{seed_source}:{masif_source}:" + os.environ.get("PYTHONPATH", "")

# Add source directories to python path so we can import modules
sys.path.append(str(seed_source))
sys.path.append(str(masif_source))

# work in protein_interaction_nn_dir
os.chdir(protein_interaction_nn_dir)

config_path = protein_interaction_nn_dir / "configs" / "protein_interaction_v1.yaml"


def submit_sbatch(slurm_script, *args):
    cmd = ["sbatch", str(slurm_script), *[str(a) for a in args]]
    print("Submitting:", " ".join(cmd))
    proc = subprocess.run(cmd, check=True, capture_output=True, text=True)
    stdout = (proc.stdout or "").strip()
    stderr = (proc.stderr or "").strip()
    if stdout:
        print(stdout)
    if stderr:
        print(stderr)
    m = re.search(r"Submitted batch job\s+(\d+)", stdout)
    if not m:
        raise RuntimeError(f"Could not parse job id from sbatch output: {stdout!r}")
    return m.group(1)


def wait_for_job(job_id, poll_s=15):
    print(f"Waiting for job {job_id} ...")
    while True:
        q = subprocess.run(
            ["squeue", "-h", "-j", str(job_id), "-o", "%T"],
            check=True,
            capture_output=True,
            text=True,
        )
        state = (q.stdout or "").strip()
        if not state:
            break
        print(f"  state={state}")
        time.sleep(poll_s)

    acct = subprocess.run(
        ["sacct", "-n", "-P", "-j", str(job_id), "--format=JobIDRaw,State,ExitCode"],
        check=True,
        capture_output=True,
        text=True,
    )
    lines = [ln.strip() for ln in (acct.stdout or "").splitlines() if ln.strip()]
    status = "UNKNOWN"
    exit_code = ""
    for ln in lines:
        fields = ln.split("|")
        if fields and fields[0] == str(job_id):
            status = fields[1]
            exit_code = fields[2] if len(fields) > 2 else ""
            break
    print(f"Job {job_id}: state={status}, exit={exit_code}")
    return status, exit_code


def show_job_log_tail(job_id, prefix="", lines=120):
    matches = sorted((protein_interaction_nn_dir / "exelogs").glob(f"{prefix}*{job_id}.out"))
    if not matches:
        print(f"No log file found for job {job_id} with prefix '{prefix}'.")
        return
    log_path = matches[-1]
    print(f"Log: {log_path}")
    out = subprocess.run(["tail", "-n", str(lines), str(log_path)], check=True, capture_output=True, text=True)
    print(out.stdout)


print("Notebook dir:", protein_interaction_nn_dir)
print("Repo root:", repo_root)
print("Config:", config_path)

Notebook dir: /scratch/ymeng/masif-neosurf-ppi-nn/masif_seed_search/data/protein_interaction_nn
Repo root: /scratch/ymeng/masif-neosurf-ppi-nn
Config: /scratch/ymeng/masif-neosurf-ppi-nn/masif_seed_search/data/protein_interaction_nn/configs/protein_interaction_v1.yaml


In [2]:
# Build tiny split CSVs via Slurm entrypoint.
job_id = submit_sbatch(
    protein_interaction_nn_dir / "run_build_pairs.slurm",
    "--config", config_path,
    "--tiny",
)
state, _ = wait_for_job(job_id)
show_job_log_tail(job_id, prefix="build_pairs-")
if not state.startswith("COMPLETED"):
    raise RuntimeError(f"Build-pairs job failed: job_id={job_id}, state={state}")


Submitting: sbatch /scratch/ymeng/masif-neosurf-ppi-nn/masif_seed_search/data/protein_interaction_nn/run_build_pairs.slurm --config /scratch/ymeng/masif-neosurf-ppi-nn/masif_seed_search/data/protein_interaction_nn/configs/protein_interaction_v1.yaml --tiny
Submitted batch job 52543459
sbatch: [ESTIMATION] The estimated cost of this job is CHF 0.01
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 23.45       │ 0.05        │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 24.2        │ 0.05        │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ 

In [3]:
pairs_dir = protein_interaction_nn_dir / "splits"
summary_path = pairs_dir / "pairs_summary.json"
print(summary_path.read_text())

{
  "available_ppi_ids": 120,
  "splits": {
    "test": {
      "num_negative": 60,
      "num_positive": 20,
      "num_ppi_ids": 20,
      "num_rows": 80
    },
    "train": {
      "num_negative": 60,
      "num_positive": 20,
      "num_ppi_ids": 20,
      "num_rows": 80
    },
    "val": {
      "num_negative": 30,
      "num_positive": 10,
      "num_ppi_ids": 10,
      "num_rows": 40
    }
  }
}


In [4]:
# Dataset and batch sanity check.
from protein_interaction.utils import load_config
from protein_interaction.dataset import read_pairs_csv, ProteinPairDataset, batch_iterator, sanity_check_batch

cfg = load_config(str(config_path))
pairs_train = pairs_dir / "pairs_train.csv"
records = read_pairs_csv(str(pairs_train))
print("Num train records:", len(records))

ds = ProteinPairDataset(cfg, records, seed=cfg.get("seed", 42))
batch = next(batch_iterator(ds, batch_size=2, descriptor_dim=cfg["model"]["descriptor_dim"], shuffle=False))
sanity_check_batch(batch)

print("query_desc:", batch["query_desc"].shape)
print("query_xyz:", batch["query_xyz"].shape)
print("query_mask:", batch["query_mask"].shape)
print("matched_desc:", batch["matched_desc"].shape)
print("matched_xyz:", batch["matched_xyz"].shape)
print("matched_mask:", batch["matched_mask"].shape)
print("labels:", batch["labels"].reshape(-1))

Num train records: 80
query_desc: (2, 256, 80)
query_xyz: (2, 256, 3)
query_mask: (2, 256)
matched_desc: (2, 1024, 80)
matched_xyz: (2, 1024, 3)
matched_mask: (2, 1024)
labels: [1. 0.]


In [5]:
# Overfit run: tiny data, many epochs, no early stop.
import yaml
from datetime import datetime

cfg = load_config(str(config_path))
cfg["train"]["epochs"] = 80
cfg["train"]["batch_size"] = 8
cfg["train"]["early_stop_patience"] = 80  # effectively disabled
cfg["data"]["tiny_subset"]["enabled"] = True
cfg["data"]["tiny_subset"]["max_pos_per_split"] = 8  # very small
cfg["data"]["negatives_per_positive"] = 1              # easier memorization

# Use a shared filesystem path so Slurm compute nodes can read the config.
tmp_cfg_dir = protein_interaction_nn_dir / "tmp_configs"
tmp_cfg_dir.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
tmp_cfg_path = tmp_cfg_dir / f"protein_interaction_v1_overfit_{ts}.yaml"
with open(tmp_cfg_path, "w") as f:
    yaml.safe_dump(cfg, f)

tmp_cfg_path = str(tmp_cfg_path)
train_job_id = submit_sbatch(
    protein_interaction_nn_dir / "run_train_v1.slurm",
    "--config",
    tmp_cfg_path,
)
train_state, _ = wait_for_job(train_job_id)
show_job_log_tail(train_job_id, prefix="train_v1-")
if not train_state.startswith("COMPLETED"):
    raise RuntimeError(f"Train job failed: job_id={train_job_id}, state={train_state}")

print("Temporary config:", tmp_cfg_path)

Submitting: sbatch /scratch/ymeng/masif-neosurf-ppi-nn/masif_seed_search/data/protein_interaction_nn/run_train_v1.slurm --config /scratch/ymeng/masif-neosurf-ppi-nn/masif_seed_search/data/protein_interaction_nn/tmp_configs/protein_interaction_v1_overfit_20260320_155217.yaml
Submitted batch job 52543466
sbatch: [ESTIMATION] The estimated cost of this job is CHF 0.18
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 23.45       │ 0.2         │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 24.2        │ 0.2         │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and th

RuntimeError: Train job failed: job_id=52543466, state=OUT_OF_MEMORY

In [ ]:
# Evaluate overfit behavior on train + test via Slurm entrypoint.
for split in ["train", "test"]:
    eval_job_id = submit_sbatch(
        protein_interaction_nn_dir / "run_eval_v1.slurm",
        "--config",
        tmp_cfg_path,
        "--split",
        split,
    )
    eval_state, _ = wait_for_job(eval_job_id)
    show_job_log_tail(eval_job_id, prefix="eval_v1-")
    if not eval_state.startswith("COMPLETED"):
        raise RuntimeError(f"Eval job failed: split={split}, job_id={eval_job_id}, state={eval_state}")

eval_dir = Path(cfg["eval"]["output_dir"])
if not eval_dir.is_absolute():
    eval_dir = repo_root / eval_dir

for split in ["train", "test"]:
    metrics_path = eval_dir / f"metrics_{split}.json"
    print(f"\n== {split.upper()} ==")
    print(metrics_path.read_text())